# Tome 10, Chapter 8: Concentration of measure (Python companion)

Exercises first, solutions at the end. We test the Efron–Stein and McDiarmid bounds on a balls-in-bins count, look at the thin Gaussian shell and at maxima of correlated Gaussians, measure longest increasing subsequences, watch the norm of a random sign matrix refuse to fluctuate, and put the **Johnson–Lindenstrauss** lemma to work as a computer science tool: compressing high-dimensional data before a nearest-neighbour search. Requires `numpy` and `scipy`.

In [ ]:
import math, bisect
import numpy as np
from scipy import stats
rng = np.random.default_rng(80)

## Exercises

**P8.1 (★) Empty bins.** Throw $m=n=10^4$ balls into bins and let $F$ be the number of empty bins. Estimate $\operatorname{Var}F$ by simulation and compare with the exact variance (compute it from indicators) and with the Efron–Stein bound (estimate $\frac12\sum_i\mathbb E(F-F^{(i)})^2$ by simulation: move ball $i$ to a fresh random bin). How do the McDiarmid tail bound and the true tail compare at $t=3\sigma$?

**P8.2 (★) The thin shell.** For $n=1,10,100,1000,10^4$, compute $\mathbb E|Z|$ exactly ($\sqrt2\,\Gamma(\frac{n+1}2)/\Gamma(\frac n2)$) and compare with $\sqrt n-\frac1{4\sqrt n}$; estimate $\operatorname{Var}|Z|$. What is its limit?

**P8.3 (★★) Maxima of Gaussians.** For $m=10^2,10^3,10^4$, simulate $\max_iG_i$ for (a) independent standard Gaussians and (b) equicorrelated ones, $G_i=\sqrt\rho\,Z_0+\sqrt{1-\rho}\,Z_i$ with $\rho=\frac12$. Report the mean, compared with $\sqrt{2\log m}$, and the standard deviation, compared with the bound $1$ of Theorem 8.6.

**P8.4 (★★) CS application: compressing data for nearest-neighbour search.** Build $N=2000$ sparse "documents" in dimension $d=20\,000$ (each a vector of word counts with about $100$ nonzero entries, drawn from a Zipf-like vocabulary). Project with a $k\times d$ Gaussian matrix and with a $\pm1$ matrix (Achlioptas), for $k=50,100,200,500,1000$. For each $k$, report (a) the worst relative distortion of squared distances over $5000$ random pairs, (b) the fraction of documents whose nearest neighbour is unchanged by the projection. Compare with the $k$ required by Theorem 8.10 for $\varepsilon=0.1$ and $0.2$.

**P8.5 (★★) Ulam's problem.** Compute $L_n$ by patience sorting for $n=10^2,10^3,10^4,10^5$ ($500$ samples each, fewer for the largest). Compare $\mathbb EL_n$ with $2\sqrt n$ and with $2\sqrt n-1.7711\,n^{1/6}$ (the Baik–Deift–Johansson correction, $-1.7711$ being the mean of the Tracy–Widom law), and the standard deviation with $0.9018\,n^{1/6}$. Check $\operatorname{Var}L_n\le\mathbb EL_n$.

**P8.6 (★★★) The norm of a random sign matrix.** Let $M$ be an $n\times n$ matrix of independent fair signs. Its operator norm is a convex, $2$-Lipschitz function of the $n^2$ variables $X_{ij}=\frac{1+M_{ij}}2\in\{0,1\}$ (Euclidean norm on $\R^{n^2}$). By Exercise 8.6, it fluctuates by $O(1)$. Check by simulation for $n=50,100,200,400$ that $\|M\|/\sqrt n\to2$ while the standard deviation of $\|M\|$ stays bounded.

## Solutions

### P8.1
**How to find it.** $F=\sum_j\mathbf 1\{\text{bin }j\text{ empty}\}$, so $\operatorname{Var}F=n q_1+n(n-1)q_2-(nq_1)^2$ with $q_1=(1-\frac1n)^m$ and $q_2=(1-\frac2n)^m$. For Efron–Stein, moving ball $i$ changes $F$ by $-1$, $0$ or $+1$: $+1$ if it leaves a bin where it was alone, $-1$ if it lands in an empty bin.

In [ ]:
m = n = 10**4
q1, q2 = (1 - 1 / n)**m, (1 - 2 / n)**m
var_exact = n * q1 + n * (n - 1) * q2 - (n * q1)**2
T = 2000
F = np.empty(T); ES = np.empty(T)
for t in range(T):
    b = rng.integers(0, n, m)
    cnt = np.bincount(b, minlength=n)
    F[t] = np.sum(cnt == 0)
    newb = rng.integers(0, n, m)                       # each ball i moved alone to newb[i]
    leaves_alone = (cnt[b] == 1)                       # its bin becomes empty
    same = (newb == b)
    delta = np.where(same, 0, leaves_alone.astype(int) - (cnt[newb] == 0).astype(int))
    ES[t] = 0.5 * np.sum(delta**2)
print(f"E F = {F.mean():.1f} (exact {n * q1:.1f});  Var F: simulated {F.var():.1f}, exact {var_exact:.1f};  Efron-Stein bound {ES.mean():.1f}")
sig = math.sqrt(var_exact); t = 3 * sig
print(f"t = 3 sigma = {t:.0f}: McDiarmid bound {2 * math.exp(-2 * t**2 / m):.3f};  Gaussian approximation {2 * stats.norm.sf(3):.4f}")

Efron–Stein overestimates the variance by a factor of about $2.4$: it adds up the effects of moving each ball separately, while these effects are negatively correlated across balls. McDiarmid's bound at $3\sigma$ is $0.35$, against a true tail of about $0.003$: it only sees $C=m$, not the true variance $\approx0.1\,m$. Bounded differences give the right scale $\sqrt m$ but not the right constant.

### P8.2
**How to find it.** $|Z|$ has the chi law; use `scipy.special.gammaln` to avoid overflow. $\operatorname{Var}|Z|=n-(\mathbb E|Z|)^2$.

In [ ]:
from scipy.special import gammaln
for n in (1, 10, 100, 1000, 10**4):
    EZ = math.sqrt(2) * math.exp(gammaln((n + 1) / 2) - gammaln(n / 2))
    print(f"n = {n:6d}: E|Z| = {EZ:.6f},  sqrt(n) - 1/(4 sqrt n) = {math.sqrt(n) - 1 / (4 * math.sqrt(n)):.6f},  Var|Z| = {n - EZ**2:.6f}")

$\operatorname{Var}|Z|\to\frac12$: by the central limit theorem, $|Z|^2=n+\sqrt{2n}\,N$ approximately, so $|Z|\approx\sqrt n+\frac N{\sqrt2}$. The bound $1$ of Theorem 8.6 is off by a factor $2$, in every dimension.

### P8.3
**How to find it.** Generate in blocks to save memory. For the equicorrelated case, $\max_iG_i=\sqrt\rho Z_0+\sqrt{1-\rho}\max_iZ_i$: the common part adds variance $\rho$ but does not change the mean.

In [ ]:
T = 4000
for m in (10**2, 10**3, 10**4):
    Mi = np.array([rng.standard_normal(m).max() for _ in range(T)])
    rho = 0.5
    Mc = math.sqrt(rho) * rng.standard_normal(T) + math.sqrt(1 - rho) * np.array([rng.standard_normal(m).max() for _ in range(T)])
    print(f"m = {m:6d}: sqrt(2 log m) = {math.sqrt(2 * math.log(m)):.3f} | independent: mean {Mi.mean():.3f}, sd {Mi.std():.3f} | rho = 1/2: mean {Mc.mean():.3f}, sd {Mc.std():.3f}")

The standard deviation stays below $1$ in all cases; for independent variables it even decreases, like $1/\sqrt{2\log m}$ (the Gumbel scaling of extreme value theory). With correlations the mean drops and the standard deviation rises towards $\sqrt\rho$, but never above the bound.

### P8.4
**How to find it.** Draw word indices from a Zipf law truncated to $d$ words, build a sparse matrix with `np.add.at`, and project with dense matrices (a $1000\times20\,000$ Gaussian matrix is $160$ MB in float64, so use float32 and project in blocks of rows). For nearest neighbours use $|x-y|^2=|x|^2+|y|^2-2x\cdot y$.

In [ ]:
N, d, nnz = 2000, 20000, 100
ranks = np.arange(1, d + 1); pz = 1 / ranks**1.1; pz /= pz.sum()
X = np.zeros((N, d), dtype=np.float32)
for i in range(N):
    words = rng.choice(d, size=nnz, p=pz)
    np.add.at(X[i], words, 1.0)

def sqdists(Y):
    sq = (Y.astype(np.float64)**2).sum(axis=1)
    D = sq[:, None] + sq[None, :] - 2 * (Y.astype(np.float64) @ Y.T.astype(np.float64))
    np.fill_diagonal(D, np.inf)
    return np.maximum(D, 0)

D0 = sqdists(X)
nn0 = D0.argmin(axis=1)
pairs = rng.integers(0, N, (5000, 2)); pairs = pairs[pairs[:, 0] != pairs[:, 1]]
print("k needed by Theorem 8.10: eps = 0.1:", math.ceil(4 * math.log(N) / (0.1**2 / 2 - 0.1**3 / 3)),
      "  eps = 0.2:", math.ceil(4 * math.log(N) / (0.2**2 / 2 - 0.2**3 / 3)))
for k in (50, 100, 200, 500, 1000):
    for kind in ("gauss", "signs"):
        Y = np.zeros((N, k), dtype=np.float32)
        for r0 in range(0, k, 100):
            r1 = min(k, r0 + 100)
            if kind == "gauss":
                A = rng.standard_normal((r1 - r0, d)).astype(np.float32)
            else:
                A = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=(r1 - r0, d))
            Y[:, r0:r1] = X @ A.T
        Y /= math.sqrt(k)
        D1 = sqdists(Y)
        ratio = D1[pairs[:, 0], pairs[:, 1]] / D0[pairs[:, 0], pairs[:, 1]]
        same_nn = np.mean(D1.argmin(axis=1) == nn0)
        print(f"k = {k:4d} {kind:5s}: worst distortion over pairs {np.max(np.abs(ratio - 1)):.3f},  nearest neighbour kept {same_nn:.3f}")

The worst distortion over $5000$ pairs falls like $k^{-1/2}$ and is about $0.17$ at $k=1000$, below the $k=1755$ that the theorem requires for $\varepsilon=0.2$ and $N=2000$: the union bound over all pairs and the constant $4$ are pessimistic. Sign matrices behave like Gaussian ones. Nearest neighbours are harder: distances to the first and second neighbours are often within a few percent of each other, so exact nearest-neighbour preservation needs a larger $k$ than distance preservation. In practice one projects, retrieves a short list of candidates, and re-ranks them with exact distances.

### P8.5
**How to find it.** Patience sorting: scan the sequence, keep the smallest possible tail of an increasing subsequence of each length, and update by binary search; the number of piles is $L_n$. It runs in $O(n\log n)$.

In [ ]:
def lis_length(seq):
    tails = []
    for v in seq:
        j = bisect.bisect_left(tails, v)
        if j == len(tails):
            tails.append(v)
        else:
            tails[j] = v
    return len(tails)

for n, T in ((10**2, 500), (10**3, 500), (10**4, 300), (10**5, 100)):
    Ls = np.array([lis_length(rng.random(n).tolist()) for _ in range(T)])
    print(f"n = {n:6d}: E L = {Ls.mean():8.2f}, 2 sqrt(n) = {2 * math.sqrt(n):8.2f}, BDJ {2 * math.sqrt(n) - 1.7711 * n**(1 / 6):8.2f};"
          f"  sd = {Ls.std():.2f} vs 0.9018 n^(1/6) = {0.9018 * n**(1 / 6):.2f};  Var/E = {Ls.var() / Ls.mean():.4f}")

The mean matches the Baik–Deift–Johansson approximation to within a fraction of a unit already at $n=1000$, and the standard deviation grows like $n^{1/6}$: about $6$ at $n=10^5$, where Talagrand's bound gives the scale $n^{1/4}\approx18$ and Efron–Stein $\sqrt{\mathbb EL_n}\approx25$. The ratio $\operatorname{Var}L_n/\mathbb EL_n$ tends to $0$: Proposition 8.2 is true but far from sharp.

### P8.6
**How to find it.** `np.linalg.norm(M, 2)` computes the largest singular value. The bound of Exercise 8.6 with Lipschitz constant $2$ gives $P(|\|M\|-\text{median}|\ge t)\le4e^{-t^2/16}$.

In [ ]:
for n in (50, 100, 200, 400):
    T = 200 if n < 400 else 60
    norms = np.array([np.linalg.norm(rng.choice([-1.0, 1.0], size=(n, n)), 2) for _ in range(T)])
    print(f"n = {n:4d}: E||M||/sqrt(n) = {norms.mean() / math.sqrt(n):.4f},  sd ||M|| = {norms.std():.3f}")

$\|M\|/\sqrt n$ approaches $2$ (the edge of the semicircle law, a theorem of random matrix theory), and the standard deviation of $\|M\|$ stays near $0.3$ (its slow decrease, like $n^{-1/6}$, is invisible over this range), although $\|M\|$ itself grows like $2\sqrt n$. Talagrand's inequality predicts $O(1)$; the truth, $O(n^{-1/6})$, is again the Tracy–Widom scale.